# 02 — Baseline CNN

Train the small CNN to validate the end-to-end training pipeline.  
For a full run use `python scripts/train_baseline.py` from `model/`; this notebook does a **short smoke run** (2 epochs).

**Prerequisite:** model package installed in the active kernel (see `model/README.md`).

In [1]:
import sys
from pathlib import Path


def _add_model_src() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "model" / "src"
        if candidate.is_dir():
            s = str(candidate)
            if s not in sys.path:
                sys.path.insert(0, s)
            return base
    raise RuntimeError(f"Cannot find model/src from {cwd}. Run Jupyter from the repo root.")


REPO_ROOT = _add_model_src()
print("Repo root:", REPO_ROOT)

Repo root: /Users/sandinosaso/repos/pathsight


In [2]:
from model_service.config import ModelServiceConfig
from model_service.data.loaders import build_pcam_datasets
from model_service.evaluation.plots import plot_history
from model_service.training.baseline import build_baseline_cnn
from model_service.training.callbacks import default_callbacks
from model_service.training.train import run_training

In [3]:
cfg = ModelServiceConfig()
cfg.data.batch_size = 32
cfg.train.epochs = 2   # smoke run — increase for real training

train_ds, val_ds, _, _ = build_pcam_datasets(
    cfg, download=True, use_efficientnet_preprocess=False
)
model = build_baseline_cnn()
model.summary()

Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,817 (432.88 KB)

 Trainable params: 110,817 (432.88 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Absolute paths so saving works wherever Jupyter was launched
models_dir   = REPO_ROOT / "artifacts" / "models"
metrics_dir  = REPO_ROOT / "artifacts" / "metrics"
figures_dir  = REPO_ROOT / "artifacts" / "figures"
for d in [models_dir, metrics_dir, figures_dir]:
    d.mkdir(parents=True, exist_ok=True)

cbs = default_callbacks(
    models_dir  / "baseline_nb.keras",
    metrics_dir / "baseline_nb.csv",
    monitor="val_auc",
    mode="max",
    early_stopping_patience=3,
)

hist = run_training(model, train_ds, val_ds, epochs=cfg.train.epochs, callbacks=cbs)
plot_history(hist, figures_dir / "baseline_nb_history.png")
print("Training complete. Best checkpoint:", models_dir / "baseline_nb.keras")

Epoch 1/2
8192/8192 ━━━━━━━━━━━━━━━━━━━━ 3672s 448ms/step - accuracy: 0.8099 - auc: 0.8907 - loss: 0.4177 - precision: 0.8047 - recall: 0.8183 - val_accuracy: 0.8046 - val_auc: 0.9006 - val_loss: 0.4124 - val_precision: 0.8419 - val_recall: 0.7496
Epoch 2/2
8192/8192 ━━━━━━━━━━━━━━━━━━━━ 3704s 452ms/step - accuracy: 0.8615 - auc: 0.9352 - loss: 0.3253 - precision: 0.8588 - recall: 0.8653 - val_accuracy: 0.8348 - val_auc: 0.9246 - val_loss: 0.3659 - val_precision: 0.8793 - val_recall: 0.7759
Training complete. Best checkpoint: /Users/sandinosaso/repos/pathsight/artifacts/models/baseline_nb.keras
